<a href="https://colab.research.google.com/github/mybenkhadda/GEMS---drivendata/blob/main/notebooks/colab/GEMS_G06_GEOLOGICAL_FEATURE_STORE_AND_ML_BENCHMARK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GEMS G06 - geological feature store & multi-task ML benchmark (code rev g06.1-r1)

**Question:** can models trained on carefully engineered geological observations discover fault evidence that W2 has not learned?

* One reusable, fold-safe **feature store** (tiled GeoTIFF groups + Parquet registry) built once from the 19-band GeoDAWN raster.
* Datasets: D1 pixels (one label-independent pool, relabelled per training visibility), D2 patches, D3 candidate lineaments, D4 fragment pairs.
* Benchmarks: pixel RF / LightGBM / XGBoost (geology only), segment classifiers (S1 geology only, S2 W2-aware), optional compact U-Net, cross-fitted meta-models vs W2 - all under the canonical official-proxy protocol (FastScorer + M7 keys, fixed policy).
* Outputs: `MyDrive/GEMS_2/outputs/g06_geological_feature_store/<run_id>/` (run id = data-engineering identity; training changes never rebuild the store; replaced outputs are archived under `history/`).
* **Nothing is uploaded.** Research rasters are tagged `ROLE=RESEARCH_NOT_A_SUBMISSION`; an export is at best `READY_FOR_MANUAL_UPLOAD` after strict validation + parity gates.

Docs: `docs/g06/README.md`, `COLAB_EXECUTION_GUIDE.md`, `VALIDATION_PROTOCOL.md`, `SESSION_HANDOFF.md`.

## 00 - Scientific objectives and model hypotheses

| id | hypothesis | test |
|---|---|---|
| H1 | Engineered potential-field / structural / texture features carry fault evidence beyond the raw bands | G06-P1 (raw) -> P2 -> P3 / P4 -> P5 feature-family ablation, V1 pooled DTI at the fixed policy |
| H2 | A geology-only model finds systems W2 misses | systems recovered by G06-P5 but not by W2 (`evaluation/system_recovery.csv`, `evaluation/complementarity.csv`) |
| H3 | Lineament-level reasoning (shape, orientation agreement, cross-trace field step, network) adds to pixel evidence | G06-S1 vs G06-P5; G06-S2 vs W2 |
| H4 | G06 complements W2 | G06-M1 rank average (no fit, STRICT_LABEL_FREE) and residual / MoE meta-models (SCREENING_ONLY) vs W2 |
| H5 | Results transfer geographically | V3 (geographic folds) next to V1; worst zone with every pooled number |

Rules: the primary metric is pooled DTI from summed components under ONE fixed policy (`evaluation.primary`); accuracy / ROC-AUC / PR-AUC are diagnostics only; a screening score is never a nested DTI; historical G05 numbers (0.282538 / 0.242156 / 0.130868) and the public score (0.0942) are references from other protocols.

## 01 - CONFIG / RUN

`CONFIG` = where / how; `RUN` = which stages. First run = data preparation (defaults below). Second run = classical models: set `TRAIN_RANDOM_FOREST` / `TRAIN_LIGHTGBM` / `EVALUATE` to True - the feature store, splits and datasets are reused from the validated cache (`REUSED_VALID_CACHE`). Exact configurations per run: `docs/g06/COLAB_EXECUTION_GUIDE.md`.

In [ ]:
CONFIG = {
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'main',                       # pin the pushed commit SHA for a reproducible run
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',
    'output_subdir': 'outputs/g06_geological_feature_store',
    'seed': 2026,
    'tile_px': 512,
    'halo_px': 64,                       # minimum; the effective halo (operator support) is recorded in the schema
    'use_gpu': False,                    # True only for TRAIN_PATCH_UNET (Runtime -> GPU)
    'reuse_cached': True,
    'strict_provenance': True,
    'sync_to_drive': True,
    'save_figures': True,
    'run_synthetic_tests': True,         # pytest tests/test_g06_*.py in Colab; TESTS_FAILED blocks training
    'run_id': None,                      # None = deterministic data-engineering identity
    'run_id_mode': 'data',               # 'commit' = a new run directory per commit (strict)
}
RUN = {
    'PREFLIGHT': True,
    'RAW_EDA': True,
    'FEATURE_ENGINEERING': True,
    'FEATURE_ATLAS': True,
    'BUILD_FEATURE_STORE': True,
    'BUILD_SPLITS': True,
    'BUILD_PIXEL_DATASET': True,
    'BUILD_SEGMENT_DATASET': True,

    'TRAIN_RANDOM_FOREST': False,
    'TRAIN_LIGHTGBM': False,
    'TRAIN_XGBOOST': False,
    'TRAIN_SEGMENT_MODEL': False,
    'TRAIN_PATCH_UNET': False,
    'TRAIN_META_MODEL': False,

    'EVALUATE': False,
    'FULL_INFERENCE': False,
    'EXPORT_SUBMISSION': False,

    'REPORT': True,
}
OVERRIDES = {'feature_store': {}, 'benchmarks': {}}   # deep-merged into configs/g06/*.yaml
PRIVATE_REPO = False
EXPECT = {'code_rev': 'g06.1-r1', 'commit': None}

## 02 - Drive mount, repository checkout, dependencies

Restart the session before re-running this cell if `gems_bench` was already imported.

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = 'https://github.com/' + CONFIG['repo'] + '.git'

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    hdr = base64.b64encode(('x-access-token:' + (userdata.get('GITHUB_TOKEN') or '')).encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + hdr}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode:
        raise RuntimeError('git ' + ' '.join(args[:4]) + ' failed:\n' + r.stderr[-1500:])
    return r

if not Path(CODE, '.git').exists():
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
if git('-C', CODE, 'rev-parse', '--verify', '-q', 'refs/remotes/origin/' + ref, check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'g06_colab_active', 'origin/' + ref)
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported: Runtime -> Restart session, then run from the top')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', CODE + '[colab,m9]', 'pytest'], check=True)
sys.path.insert(0, CODE)
os.chdir(CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
print('HEAD', HEAD)

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
import gems_bench.g06 as G06
from gems_bench.g06 import config as GC, feature_store as FST, raster_io as RIO, visualization as VIZ
from gems_bench.g06.pipeline import G06Pipeline
assert G06.G06_CODE_REV == EXPECT['code_rev'], f'code rev {G06.G06_CODE_REV} != {EXPECT["code_rev"]}'
assert not EXPECT['commit'] or HEAD.startswith(EXPECT['commit']), f'HEAD {HEAD} != {EXPECT["commit"]}'
cfg = GC.load_config(overrides={'notebook': CONFIG, 'run': RUN, **{k: v for k, v in OVERRIDES.items() if v}})
P = G06Pipeline(cfg, RUN, HEAD)
OUT = P.root

def show(rel, cols=None, n=30):
    p = OUT / rel
    if not p.exists():
        print('ABSENT:', rel); return None
    df = pd.read_csv(p) if p.suffix == '.csv' else pd.read_parquet(p)
    display(df[[c for c in (cols or df.columns) if c in df]].head(n)); return df

def js(rel, n=2500):
    d = P.ws.read_json(rel)
    print(json.dumps(d, indent=1, default=str)[:n] if d is not None else f'ABSENT: {rel}')

def img(rel):
    p = OUT / rel
    display(Image(str(p))) if p.exists() else print('ABSENT:', rel)

print('run id', P.run_id)
print('local', OUT, '| Drive', P.ws.drive_path())
print('sources', {k: str(v) for k, v in P.src.items()})

## 03 - Full artifact preflight

**Inputs:** the 19-band raster, fault labels, template, Drive artifacts (folds, splits, P6 W2 maps, M9 CNN maps, G01 / G02 / G05 / G05.1 / G05.2 outputs). **Operation:** synthetic tests, exact band resolution (fails closed), artifact discovery and grid compatibility, frozen-engine attach. **Outputs:** `inventory/dependency_inventory.csv`, `band_inventory.csv`, `artifact_compatibility.csv`, `missing_dependencies.json`. **Interpretation:** a FAIL on a mandatory dependency blocks the stages that need it; optional artifacts stay UNAVAILABLE (never zero-filled).

In [ ]:
P.run_tests()
show('audit/tests.csv')
P.preflight()
show('inventory/dependencies.csv', ['dependency', 'mandatory', 'status', 'detail'], n=40)
js('inventory/missing_dependencies.json')

## 04 - Raster and grid validation

Grid contract 3730 x 3292, EPSG:32611, 100 m, 19 bands. Every band is resolved by its **exact** identifier (`iso_grav_anom` can never resolve to `iso_grav_anom_hg`, the G05 failure). Check the `description` column against the declared units in `configs/g06/feature_store.yaml`.

In [ ]:
show('inventory/band_inventory.csv', ['index', 'identifier', 'description', 'status', 'band_index', 'substring_near_misses', 'modality', 'unit_declared', 'transform'], n=25)
show('inventory/artifact_compatibility.csv', n=40)
d = P.ws.read_json('manifests/dependencies.json') or {}
print('grid', d.get('grid'))
print('engine', d.get('engine'))

## 05 - Raw 19-band exploration

Footprint (>= 50 % of bands finite), per-band coverage / moments / quantiles / constant flag (one band in memory at a time), raw atlas with mapped faults outlined (visualisation only), magnetic vs gravity comparison, missingness map. Outputs: `inventory/raw_bands.csv`, `figures/raw_atlas/`.

In [ ]:
P.raw_eda()
show('inventory/raw_bands.csv', ['band', 'index', 'modality', 'frac_finite', 'missing_in_footprint', 'min', 'q01', 'q50', 'q99', 'max', 'constant'], n=25)
for f in ('raw_bands_atlas', 'magnetic_vs_gravity', 'missingness'):
    img(f'figures/raw_atlas/{f}.png')

## 06 - Physical feature engineering (preview on one real window)

`feature_physics`: masked (normalised-convolution) gradients, orientation (doubled angle), Laplacian, Hessian eigenvalues, directional second derivatives, compressed curvature, thin edges and their multiscale persistence at 100 / 300 / 700 / 1500 m; DoG band energies; whole-raster FFT tilt / analytic signal / upward-continuation residual (global group). A Gaussian scale is a smoothing width, **not** a source depth. The preview computes the same code on one central window; the store (section 10) computes every tile.

In [ ]:
import rasterio
PREVIEW = {}

def preview(group, k=8, size=512):
    fs = cfg['feature_store']
    if not PREVIEW:
        bi = P.band_index()
        stats = P.ws.load_table('inventory/raw_bands.csv')
        aux = {'entropy_edges': P.ws.read_json('inventory/entropy_edges.json') or {},
               'median_abs': dict(zip(stats.band, stats.get('median_abs', pd.Series(1.0, index=stats.index))))}
        miss = []
        names = FST.group_names(fs, miss)
        halo = FST.effective_halo(fs)
        g = P.grid()
        r0, c0 = g.height // 2 - size // 2, g.width // 2 - size // 2
        win = RIO.TileWindow('preview', r0, r0 + size, c0, c0 + size, halo, g.height, g.width)
        with rasterio.open(P.src['features']) as ds:
            arr = RIO.read_aligned_window(ds, [bi[b] for b in fs['bands']], win)
        fields = {b: arr[j].astype(np.float64) for j, b in enumerate(fs['bands'])}
        tiled = {x: v for x, v in names.items() if FST.GROUPS[x][2] == 'tiled'}
        PREVIEW.update(names=names, stacks=FST.compute_tile(fields, fs, tiled, miss, aux, win))
    ids = PREVIEW['names'][group]
    pick = [ids[i] for i in np.linspace(0, len(ids) - 1, min(k, len(ids))).astype(int)]
    arrays = {n: PREVIEW['stacks'][group][ids.index(n)] for n in pick}
    VIZ.plot_feature_atlas(arrays, OUT / 'figures' / 'previews', f'preview_{group}', f'{group} (central {size} px window)', None, 1, 4)
    img(f'figures/previews/preview_{group}.png')

print('effective halo', FST.effective_halo(cfg['feature_store']), 'px (minimum', CONFIG['halo_px'], ')')
print({g: len(v) for g, v in FST.group_names(cfg['feature_store'], []).items()})

In [ ]:
if RUN['FEATURE_ENGINEERING']:
    for g in ('magnetic', 'gravity', 'multiscale'):
        preview(g)

## 07 - Texture extraction

`feature_texture`: masked window std / range / mean / quantiles, normalised entropy (bins = footprint quantiles, label-free), thin-edge density, gradient variability, local anisotropy, orientation histograms over 3 / 7 / 15 / 31 px windows. Invalid pixels are never read as physical zeros.

In [ ]:
if RUN['FEATURE_ENGINEERING']:
    preview('texture', k=12)

## 08 - Structural feature extraction

`feature_structure`: structure-tensor coherence and dominant axis (doubled angle), eigenvalue ratio, scale-normalised ridge / valley, along-strike continuity, cross-strike field step, truncation proxy, multiscale directional persistence, cross-modal axial agreement A = (1 + cos 2(t1 - t2)) / 2, co-located discontinuities. Curvature / truncation are **proxies, not displacement measurements**.

In [ ]:
if RUN['FEATURE_ENGINEERING']:
    preview('structural', k=12)

## 09 - Geological-context features

Regional (label-free, stored): regional magnetic / gravity texture, terrain roughness, seismicity density, asinh strain rates, basement depth and relief, distance to the footprint edge, unsupervised k-means domains. **Permitted known-fault context** (distance, density, tips, orientation, network density) is rebuilt per validation unit from its visible faults only and is never a pixel-model input (section 15 builds it for segments).

In [ ]:
if RUN['FEATURE_ENGINEERING']:
    preview('context', k=9)

## 10 - Full feature-store construction

One tiled pass (512 px tiles + effective halo) computes every tiled group; each tile is cached and verified before it counts as complete (a disconnect resumes at the first incomplete tile); groups are assembled into compressed GeoTIFFs and verified (alignment + tile read-back); global groups follow. Outputs: `features/*`, `feature_registry.parquet`, `feature_schema.json`, `feature_coverage.csv`. **Long on a first run** (CPU, tens of minutes to hours); reused afterwards.

In [ ]:
P.build_feature_store()
R = show('features/feature_registry.parquet', n=0)
if R is not None and len(R):
    display(R.groupby(['group', 'family']).agg(n=('feature_id', 'size'), usable=('usable', 'sum')))
    display(R[~R.usable.astype(bool)][['feature_id', 'group', 'frac_finite', 'std']])
    st = P.store()
    print('schema', st.schema_hash, '| halo', st.schema.get('halo_effective_px'))
    print('seam check', RIO.compare_tile_seams(st.path('magnetic'), 1, CONFIG['tile_px']))

## 11 - Geological feature atlas and QC

Per-group atlases (consistent extent), Spearman correlations of a footprint sample, fault-centred profiles (median / IQR vs distance to mapped faults - descriptive, post hoc).

In [ ]:
P.feature_atlas()
for rel in ('figures/physical_features/atlas_magnetic.png', 'figures/physical_features/atlas_magnetic_spectral.png',
            'figures/structure/atlas_structural.png', 'figures/feature_distributions/feature_correlations.png',
            'figures/structure/fault_profiles.png'):
    img(rel)

## 12 - Fold construction and leakage audit

Units: V1 (withheld complete systems, primary), V3 (geographic folds), V2 (network completion, diagnostic). Training visibilities: `V1:d`, `V3:k`, nested `V1:j|k` / `V3:j|k`, `FULL`. Every fit is checked: no scored truth of a predicted unit is visible, geographic regions never touch the evaluation zone, known context never contains the unit truth. Outputs: `splits/*`, `manifests/validation_contract.json`.

In [ ]:
P.build_splits()
show('splits/units.csv', ['unit', 'task', 'eval_px', 'scored_truth_px', 'n_systems', 'known_px'], n=20)
show('splits/fits.csv', ['fit_id', 'visible_px', 'hidden_px', 'region_px', 'visible_rule', 'region_rule', 'predicts'], n=30)
A = show('splits/isolation_audit.csv', n=0)
if A is not None and len(A): display(A.status.value_counts())

## 13 - Supervision design

Incomplete labels: POSITIVE (visible fault), IGNORE (outside region / within 300 m of a visible fault), UNLABELED (PU weight 0.3), LIKELY_BACKGROUND (>= 3 km). Hidden faults get **no** class of their own (they are unlabelled at their natural rate). Segment / connection targets are evaluation-side and used only to train on OTHER units. The map below shows the classes of fit `V1:0` on a window.

In [ ]:
if P.status.get('splits') in ('COMPLETE', 'REUSED_VALID_CACHE'):
    from gems_bench.g06.splits import build_training_visibility
    from gems_bench.g06.supervision import build_pixel_supervision
    eng = P.engine()
    vis = build_training_visibility(eng, 'V1:0', cfg['benchmarks']['validation'])
    sup = build_pixel_supervision(vis.visible, vis.region, eng.valid(), cfg['benchmarks']['supervision'])
    c = sup['classes'][1500:2300, 1200:2000].astype(float)
    VIZ.plot_feature_atlas({'classes (-1 ignore, 0 unlabelled, 1 positive, 2 likely bg)': c, 'distance to visible fault (km)': sup['target_dist_km'][1500:2300, 1200:2000]},
                           OUT / 'figures' / 'training', 'supervision_classes_V1_0', 'Supervision of fit V1:0 (window)', None, 1, 2)
    img('figures/training/supervision_classes_V1_0.png')
    del vis, sup

## 14 - Spatial and class-aware sampling (D1 pixel dataset)

One pool: label stratum (all faults thinned to 300 m, capped per system), block-stratified background, label-free evidence stratum. Per fit: rows on hidden faults from the label stratum are dropped; positives weighted per system (sqrt length); PU / likely-background / hard-negative weights; partial region balance; inner spatial folds. Features are read from the store ONCE (`pool_features.parquet`).

In [ ]:
P.build_pixel_dataset()
show('datasets/pixels/label_summary.csv', n=30)
js('datasets/dataset_manifests/pixel.json', 1200)
for f in sorted((OUT / 'figures' / 'training').glob('*.png')):
    display(Image(str(f)))

## 15 - Pixel / patch / segment dataset exports

D2 patch index (label-free), D3 candidate lineaments per unit (label-free pools: honest W2, CNN fusion, physics edge strength; geometry, geology means, orientation agreement, cross-trace field step, permitted known context, honest W2 / CNN evidence), nested known context per (outer, unit) pair, D4 fragment pairs.

In [ ]:
P.build_segment_dataset()
show('datasets/segments/segment_summary.csv', n=20)
show('datasets/patches/patch_index.parquet', n=5)
js('datasets/dataset_manifests/segment.json', 1500)

## 16 - Raw-feature Random Forest baseline (G06-P0) and all enabled pixel experiments

`train_pixel_models` runs every pixel experiment whose model flag is on (RF -> `TRAIN_RANDOM_FOREST`, LightGBM -> `TRAIN_LIGHTGBM`, XGBoost -> `TRAIN_XGBOOST`) over the primary fits (and the nested fits of `validation.nested_pixel_experiments`). Held-out predictions: `predictions/oof/<experiment>/<fp10>/`.

In [ ]:
P.train_pixel_models()
show('models/pixel_registry.csv', ['experiment', 'kind', 'families', 'fit_id', 'status', 'n_train', 'n_pos', 'best_iteration', 'seconds'], n=60)

## 17 - Engineered-feature LightGBM (G06-P1 ... P5)

Feature importance (mean gain share across fits) per experiment.

In [ ]:
for f in sorted((OUT / 'figures' / 'model_diagnostics').glob('importance_*.png')):
    print(f.name); display(Image(str(f)))

## 18 - Feature-family ablations

P1 raw -> P2 + physics -> P3 + texture / P4 + structure -> P5 full, same folds and fixed policy (filled by section 22). `delta_vs_raw_lgbm` = gain over the raw-band LightGBM.

In [ ]:
show('evaluation/ablations.csv', n=40)

## 19 - Segment classifier (G06-S1 geology only, G06-S2 W2-aware)

Cross-fitted over units (other zones of the scenario, buffer around the held-out zone); nested known context and nested G06 evidence for training units. Strictness is recorded per experiment (S1 is STRICT_NESTED only with label-free pools + complete nested inputs).

In [ ]:
P.train_segment_models()
show('models/segment_registry.csv', n=40)
js('models/segment_models.json')

## 20 - Optional patch U-Net (G06-U1)

Opt-in after the classical benchmarks; set `CONFIG['use_gpu'] = True` and a GPU runtime. Channel memmap written once; fold-specific robust scaling; PU loss weights; inner early stopping; Hann-blended tiled inference.

In [ ]:
P.train_patch_models()
show('models/patch_registry.csv')

## 21 - Cross-fitted comparison with W2 (G06-M1)

W2 reference, predeclared rank average (no fit: STRICT_LABEL_FREE), residual LightGBM on logit(W2), regularised domain MoE (both SCREENING_ONLY: W2 / CNN inputs are not nested). Complementarity: rank correlation, top-area overlap, systems new / lost vs W2.

In [ ]:
P.train_meta_models()
js('models/meta_registry.json', 1500)
show('evaluation/complementarity.csv', n=40)

## 22 - Official-proxy DTI evaluation

Every available held-out map (W2, pixel, U-Net, segment, meta) on every unit, the same eligible budget domain, reps and fracs; pooled DTI from summed components, worst zone, per zone, systems, PR-AUC / calibration (diagnostic), nested representation selection, promotion gates.

In [ ]:
P.evaluate_models()
prim = cfg['benchmarks']['evaluation']['primary']
S = show('evaluation/metrics.csv', n=0)
if S is not None and len(S):
    display(S[(S.rep == prim['rep']) & (S.frac == prim['frac'])].pivot_table(index='model', columns='scenario', values=['dti', 'worst_zone_dti']).round(4))
show('evaluation/model_comparison.csv', n=40)
show('evaluation/promotion_decisions.csv', n=20)
for f in ('model_diagnostics/model_comparison', 'model_diagnostics/geographic_robustness', 'dti_frontiers/dti_frontier_V1_top', 'model_diagnostics/pr_auc_calibration'):
    img(f'figures/{f}.png')

## 23 - Fault recovery and failure atlases

Systems newly recovered / lost vs W2 for the best G06 model at the primary cell (held-out truth drawn **post hoc** only).

In [ ]:
show('evaluation/system_recovery.csv', ['model', 'unit', 'rep', 'frac', 'n_systems', 'n_recovered', 'n_recovered_w2', 'n_new', 'n_lost', 'new_ids', 'lost_ids'], n=40)
for f in sorted((OUT / 'figures' / 'fault_failures').glob('*.png')):
    print(f.name); display(Image(str(f)))

## 24 - Full-region research predictions

FULL-fit model (all mapped faults visible) predicted over the whole footprint tile by tile; train / inference parity verified on the training rows. Export (`EXPORT_SUBMISSION`) only for a strict PROMOTE_CANDIDATE with parity PASS, and even then `READY_FOR_MANUAL_UPLOAD` - never uploaded.

In [ ]:
P.reconstruct_full_region()
js('predictions/full_region/full_inference_manifest.json', 2000)
js('submissions/submission_manifest.json', 1000)

## 25 - Final benchmarking report

In [ ]:
P.create_reports()
for r in ('EXECUTION_STATUS', 'MODEL_BENCHMARK', 'NEXT_EXPERIMENT'):
    p = OUT / 'reports' / f'{r}.md'
    if p.exists(): display(Markdown(p.read_text()))

## 26 - Artifact inventory and resume instructions

Resume after a disconnect: re-run sections 01-02 and the init cell with the same `CONFIG` (same run id); every validated stage is `REUSED_VALID_CACHE`, the feature store resumes at the first incomplete tile. Earlier outputs that a changed configuration replaced are under `history/` (`history/index.csv`). `P.run()` runs every stage in order with the same `RUN` gates.

In [ ]:
display(P.stage_table())
inv = show('manifests/artifact_inventory.csv', n=0)
if inv is not None and len(inv):
    display(inv.groupby('section').agg(files=('path', 'size'), mb=('bytes', lambda b: round(b.sum() / 1e6, 1))))
show('history/index.csv')
P.final_message()
# P.run()   # alternative: every stage in order